# 上下文块头 (CCH)

## 概述

上下文块头 (CCH) 是一种创建包含更高级别上下文（例如文档级或节级上下文）的块头的方法，并在嵌入这些块头之前将这些块头添加到块中。这使得嵌入能够更准确、更完整地表示文本的内容和含义。在我们的测试中，此功能可显着提高检索质量。除了提高检索正确信息的速度之外，CCH 还降低了搜索结果中出现不相关结果的速度。这降低了法学硕士在下游聊天和生成应用程序中误解一段文本的速度。

## 动机

开发人员在使用 RAG 时面临的许多问题都可以归结为：各个块通常不包含足够的上下文来供检索系统或 LLM 正确使用。这导致无法回答问题，更令人担忧的是，出现幻觉。

这个问题的例子
- 词组经常通过隐含的指称和代词来指代它们的主题。这会导致它们在应该检索时未被检索到，或者无法被法学硕士正确理解。
- 各个块通常仅在整个部分或文档的上下文中才有意义，并且在单独阅读时可能会产生误导。

## 关键组件

#### 上下文块头
这里的想法是通过在块头前面添加更高级别的上下文到块中。该块标题可以像文档标题一样简单，也可以使用文档标题、简洁的文档摘要以及节和子节标题的完整层次结构的组合。

## 方法详细信息

#### 上下文生成
在下面的演示中，我们使用 LLM 为文档生成描述性标题。这是通过一个简单的提示来完成的，您可以在其中传递文档文本的截断版本，并要求法学硕士为文档生成描述性标题。如果您已经有足够描述性的文档标题，那么您可以直接使用它们。我们发现文档标题是块头中包含的最简单且最重要的高级上下文。

您可以在块标头中包含其他类型的上下文：
- 简洁的文档摘要
- 章节/小节标题
    - 这有助于检索系统处理文档中较大部分或主题的查询。

#### 嵌入带有块头的块
您为每个块嵌入的文本只是块标题和块文本的串联。如果您在检索期间使用重排序器，您将需要确保在那里也使用相同的串联。

#### 将块标头添加到搜索结果中
在向 LLM 呈现搜索结果时包含块标题也是有益的，因为它为 LLM 提供了更多上下文，并且使其不太可能误解块的含义。

![您的技术名称](../images/contextual_chunk_headers.svg)

## 设置

您需要为此笔记本提供 Cohere API 密钥和 OpenAI API 密钥。

# 包安装和导入

下面的单元格安装运行此笔记本所需的所有必需软件包。

In [ ]:
# Install required packages
!pip install langchain openai python-dotenv tiktoken

In [3]:
import cohere
import tiktoken
from typing import List
from openai import OpenAI
import os
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load environment variables from a .env file
load_dotenv()
os.environ["CO_API_KEY"] = os.getenv('CO_API_KEY') # Cohere API key
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY') # OpenAI API key

## 加载文档并将其分割成块
我们将在本演示中使用基本的 LangChain RecursiveCharacterTextSplitter，但您可以将 CCH 与更复杂的分块方法结合起来，以获得更好的性能。

In [ ]:
# Download required data files
import os
os.makedirs('data', exist_ok=True)

# Download the PDF document used in this notebook
!wget -O data/Understanding_Climate_Change.pdf https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf
!wget -O data/nike_2023_annual_report.txt https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/nike_2023_annual_report.txt


In [4]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

def split_into_chunks(text: str, chunk_size: int = 800) -> list[str]:
    """
    Split a given text into chunks of specified size using RecursiveCharacterTextSplitter.

    Args:
        text (str): The input text to be split into chunks.
        chunk_size (int, optional): The maximum size of each chunk. Defaults to 800.

    Returns:
        list[str]: A list of text chunks.

    Example:
        >>> text = "This is a sample text to be split into chunks."
        >>> chunks = split_into_chunks(text, chunk_size=10)
        >>> print(chunks)
        ['This is a', 'sample', 'text to', 'be split', 'into', 'chunks.']
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=0,
        length_function=len
    )
    documents = text_splitter.create_documents([text])
    return [document.page_content for document in documents]

# File path for the input document
FILE_PATH = "data/nike_2023_annual_report.txt"

# Read the document and split it into chunks
with open(FILE_PATH, "r") as file:
    document_text = file.read()

chunks = split_into_chunks(document_text, chunk_size=800)

## 生成描述性文档标题以在块头中使用

In [5]:
# Constants
DOCUMENT_TITLE_PROMPT = """
INSTRUCTIONS
What is the title of the following document?

Your response MUST be the title of the document, and nothing else. DO NOT respond with anything else.

{document_title_guidance}

{truncation_message}

DOCUMENT
{document_text}
""".strip()

TRUNCATION_MESSAGE = """
Also note that the document text provided below is just the first ~{num_words} words of the document. That should be plenty for this task. Your response should still pertain to the entire document, not just the text provided below.
""".strip()

MAX_CONTENT_TOKENS = 4000
MODEL_NAME = "gpt-4o-mini"
TOKEN_ENCODER = tiktoken.encoding_for_model('gpt-3.5-turbo')

def make_llm_call(chat_messages: list[dict]) -> str:
    """
    Make an API call to the OpenAI language model.

    Args:
        chat_messages (list[dict]): A list of message dictionaries for the chat completion.

    Returns:
        str: The generated response from the language model.
    """
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=chat_messages,
        max_tokens=MAX_CONTENT_TOKENS,
        temperature=0.2,
    )
    return response.choices[0].message.content.strip()

def truncate_content(content: str, max_tokens: int) -> tuple[str, int]:
    """
    Truncate the content to a specified maximum number of tokens.

    Args:
        content (str): The input text to be truncated.
        max_tokens (int): The maximum number of tokens to keep.

    Returns:
        tuple[str, int]: A tuple containing the truncated content and the number of tokens.
    """
    tokens = TOKEN_ENCODER.encode(content, disallowed_special=())
    truncated_tokens = tokens[:max_tokens]
    return TOKEN_ENCODER.decode(truncated_tokens), min(len(tokens), max_tokens)

def get_document_title(document_text: str, document_title_guidance: str = "") -> str:
    """
    Extract the title of a document using a language model.

    Args:
        document_text (str): The text of the document.
        document_title_guidance (str, optional): Additional guidance for title extraction. Defaults to "".

    Returns:
        str: The extracted document title.
    """
    # Truncate the content if it's too long
    document_text, num_tokens = truncate_content(document_text, MAX_CONTENT_TOKENS)
    truncation_message = TRUNCATION_MESSAGE.format(num_words=3000) if num_tokens >= MAX_CONTENT_TOKENS else ""

    # Prepare the prompt for title extraction
    prompt = DOCUMENT_TITLE_PROMPT.format(
        document_title_guidance=document_title_guidance,
        document_text=document_text,
        truncation_message=truncation_message
    )
    chat_messages = [{"role": "user", "content": prompt}]
    
    return make_llm_call(chat_messages)

# Example usage
if __name__ == "__main__":
    # Assuming document_text is defined elsewhere
    document_title = get_document_title(document_text)
    print(f"Document Title: {document_title}")

NIKE, INC. ANNUAL REPORT ON FORM 10-K


## 添加块头并测量影响
让我们看一个具体的例子来演示添加块头的影响。我们将使用 Cohere 重排序器来测量带有和不带有块头的查询的相关性。

In [13]:
def rerank_documents(query: str, chunks: List[str]) -> List[float]:
    """
    Use Cohere Rerank API to rerank the search results.

    Args:
        query (str): The search query.
        chunks (List[str]): List of document chunks to be reranked.

    Returns:
        List[float]: List of similarity scores for each chunk, in the original order.
    """
    MODEL = "rerank-english-v3.0"
    client = cohere.Client(api_key=os.environ["CO_API_KEY"])

    reranked_results = client.rerank(model=MODEL, query=query, documents=chunks)
    results = reranked_results.results
    reranked_indices = [result.index for result in results]
    reranked_similarity_scores = [result.relevance_score for result in results]
    
    # Convert back to order of original documents
    similarity_scores = [0] * len(chunks)
    for i, index in enumerate(reranked_indices):
        similarity_scores[index] = reranked_similarity_scores[i]

    return similarity_scores

def compare_chunk_similarities(chunk_index: int, chunks: List[str], document_title: str, query: str) -> None:
    """
    Compare similarity scores for a chunk with and without a contextual header.

    Args:
        chunk_index (int): Index of the chunk to inspect.
        chunks (List[str]): List of all document chunks.
        document_title (str): Title of the document.
        query (str): The search query to use for comparison.

    Prints:
        Chunk header, chunk text, query, and similarity scores with and without the header.
    """
    chunk_text = chunks[chunk_index]
    chunk_wo_header = chunk_text
    chunk_w_header = f"Document Title: {document_title}\n\n{chunk_text}"

    similarity_scores = rerank_documents(query, [chunk_wo_header, chunk_w_header])

    print(f"\nChunk header:\nDocument Title: {document_title}")
    print(f"\nChunk text:\n{chunk_text}")
    print(f"\nQuery: {query}")
    print(f"\nSimilarity without contextual chunk header: {similarity_scores[0]:.4f}")
    print(f"Similarity with contextual chunk header: {similarity_scores[1]:.4f}")

# Notebook cell for execution
# Assuming chunks and document_title are defined in previous cells
CHUNK_INDEX_TO_INSPECT = 86
QUERY = "Nike climate change impact"

compare_chunk_similarities(CHUNK_INDEX_TO_INSPECT, chunks, document_title, QUERY)


Chunk header:
Document Title: NIKE, INC. ANNUAL REPORT ON FORM 10-K

Chunk text:
Given the broad and global scope of our operations, we are particularly vulnerable to the physical risks of climate change, such 
as shifts in weather patterns. Extreme weather conditions in the areas in which our retail stores, suppliers, manufacturers, 
customers, distribution centers, offices, headquarters and vendors are located could adversely affect our operating results and 
financial condition. Moreover, natural disasters such as earthquakes, hurricanes, wildfires, tsunamis, floods or droughts, whether 
occurring in the United States or abroad, and their related consequences and effects, including energy shortages and public 
health issues, have in the past temporarily disrupted, and could in the future disrupt, our operations, the operations of our

Query: Nike climate change impact

Similarity w/o contextual chunk header: 0.10576342
Similarity with contextual chunk header: 0.92206234


这一部分显然是关于气候变化对某些组织的影响，但其中没有明确提到“耐克”。因此与查询“Nike Climate Change Impact”的相关性仅在 0.1 左右。只需将文档标题添加到块中，相似度就会高达 0.92。

# 评估结果

#### 风筝

我们根据我们创建的端到端 RAG 基准评估了 CCH，该基准称为 KITE（知识密集型任务评估）。

KITE 目前由 4 个数据集和总共 50 个问题组成。
- **AI 论文** - 约 100 篇有关 AI 和 RAG 的学术论文，以 PDF 形式从 arXiv 下载。
- **BVP Cloud 10-Ks** - 10-Ks 适用于 Bessemer Cloud Index 中的所有公司（约 70 家），采用 PDF 形式。
- **Sourcegraph 公司手册** - 约 800 个 Markdown 文件及其原始目录结构，从 Sourcegraph 的可公开访问的公司手册 GitHub [页面](https://github.com/sourcegraph/handbook/tree/main/content) 下载。
- **最高法院意见** - 2022 年任期的所有最高法院意见（从 23 年 1 月到 23 年 6 月交付），以 PDF 形式从最高法院官方[网站](https://www.supremecourt.gov/opinions/slipopinion/22)下载。

每个样本都包含真实答案。大多数样本还包括评分标准。每个问题的评分范围为 0-10，由实力雄厚的法学硕士进行评分。

我们比较使用和不使用 CCH 的性能。对于 CCH 配置，我们使用文档标题和文档摘要。两种配置之间的所有其他参数保持相同。我们使用 Cohere 3 重排序器，并使用 GPT-4o 进行响应生成。

|                         |无CCH | CCH |
|------------------------|----------|--------------|
|人工智能论文 | 4.5 | 4.5 4.7 | 4.7
| BVP 云 | 2.6 | 2.6 6.3 |
|来源图 | 5.7 | 5.7 5.8 |
|最高法院意见| 6.1 | 7.4 | 7.4
| **平均** | 4.72 | 4.72 6.04 | 6.04

我们可以看到 CCH 提高了四个数据集的性能。一些数据集有很大的改进，而另一些数据集则有很小的改进。总体平均分从4.72提高到6.04，提高了27.9%。

#### 金融基准

我们还在 FinanceBench 上评估了 CCH，其得分为 83%，而基准得分为 19%。对于该基准，我们联合测试了 CCH 和相关片段提取 (RSE)，因此我们无法准确说明 CCH 对这一结果的贡献有多大。但 CCH 和 RSE 的结合显然可以显着提高 FinanceBench 的准确性。

![](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--contextual-chunk-headers)